# Mapper — Final Hypothesis Verdicts (one-shot, Run all)
**فقط `Runtime → Run all` بزن** (بهتر است GPU روشن باشد: `Runtime → Change runtime type → T4 GPU`).
زمان تقریبی: ۱ تا ۲٫۵ ساعت. تب را باز نگه دار.
در پایان `final_results.zip` خودکار دانلود می‌شود و فایل **`FINAL_VERDICTS.csv`** + **`FINAL_REPORT.md`** حاوی حکم قطعی هر فرضیه است.

قواعد تصمیم‌گیری **قبل از اجرا** در سلول تنظیمات ثابت شده‌اند و بعد از دیدن نتایج تغییر نمی‌کنند.

In [1]:
!pip install -q kmapper sentence-transformers networkx datasets statsmodels

## 1) تنظیمات و قواعد تصمیم (ثابت)

In [2]:
import os, json, random, warnings, time, zlib, tarfile, urllib.request
from collections import Counter
from itertools import combinations
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---------- داده و مدل ----------
PER_TIER_MAX = 4000
TEST_FRAC    = 0.30
MODELS  = ["sentence-transformers/all-MiniLM-L6-v2", "BAAI/bge-small-en-v1.5", "sentence-transformers/all-mpnet-base-v2"]
PRIMARY = MODELS[0].split("/")[-1]
TIER_SEEDS = [42, 43, 44, 45, 46]     # تکرار نمونه‌گیری/split برای مدل اصلی
REAL_SEEDS = [1, 2, 3, 4, 5]          # تکرار برای آزمون اختلاف واقعی annotator ها

# ---------- Mapper (مطابق مقاله پایه) ----------
N_CUBES, OVERLAP, MIN_CLUSTER, PURE = 40, 0.30, 2, 0.90
INFORMATIVE = ["pca_1d", "centroid_1d", "eccentricity_1d", "density_1d"]   # lens های اصلی؛ دو lens تصادفی فقط کنترل منفی هستند
N_PERM_MAIN, N_PERM_SEED, N_PERM_SENS, N_PERM_ABL = 999, 199, 199, 499
N_BOOT = 500
N_PERM_REAL = 50

# ---------- قواعد تصمیم (از پیش تعیین‌شده) ----------
RULES = dict(
  ALPHA = 0.05,
  H1H2_LENS_MAJORITY = 3,        # یک شرط (مدل×تسک×سطح) وقتی «گذر» می‌کند که >=3 از 4 lens اصلی: اثر مثبت و q<.05
  H1H2_COND_FRACTION = 2/3,      # فرضیه وقتی تأیید می‌شود که >= 2/3 شرط‌ها در *هر* سه مدل گذر کنند
  H3A_FRACTION = 0.90,           # >=90% سلول‌ها: balanced-accuracy بالاتر از هر دو baseline با CI بالای صفر
  H3B_FRACTION = 2/3,            # >=2/3 (مدل×تسک): odds ratio توافق >1 و q<.05
  H4_RHO_MIN = 0.50, H4_SAMEDIR_MIN = 0.70,   # همبستگی رتبه‌ای >=0.5 با CI پایین>0 و هم‌جهتی >=70% برای هر سه جفت مدل و هر دو معیار
  H5_W_MAX = 0.30,               # lens-invariance: q>=.05 و Kendall W<0.3 برای همه مدل×معیارها
  H6_POS_FRACTION = 0.80,        # >=80% پیکربندی‌های شبکه‌ی پارامتر: اثر مثبت (برای CP و EA)
  H7D_TASK_MIN = 2,              # حداقل ۲ از ۳ تسک: (High-Low) با CI بالای صفر برای CP و EA
  H8_POS_FRACTION = 0.80, H8_TASK_MIN = 2,
)
OUT = "final_results"; CACHE = "emb_cache"
os.makedirs(OUT, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
for _f in os.listdir(OUT):
    if _f.endswith((".csv", ".png", ".md")): os.remove(os.path.join(OUT, _f))
print("RULES:", json.dumps(RULES, indent=1))

RULES: {
 "ALPHA": 0.05,
 "H1H2_LENS_MAJORITY": 3,
 "H1H2_COND_FRACTION": 0.6666666666666666,
 "H3A_FRACTION": 0.9,
 "H3B_FRACTION": 0.6666666666666666,
 "H4_RHO_MIN": 0.5,
 "H4_SAMEDIR_MIN": 0.7,
 "H5_W_MAX": 0.3,
 "H6_POS_FRACTION": 0.8,
 "H7D_TASK_MIN": 2,
 "H8_POS_FRACTION": 0.8,
 "H8_TASK_MIN": 2
}


## 2) ورودی/خروجی داده و embedding

In [3]:
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def load_socchem_raw():
    for name in ["tasksource/social-chemestry-101", "leondz/trustgpt_social_chemistry"]:
        try:
            ds = load_dataset(name)
            df = pd.concat([ds[k].to_pandas() for k in ds.keys()], ignore_index=True)
            print("loaded:", name, df.shape); return df
        except Exception as e:
            print("failed:", name, repr(e)[:150])
    raise RuntimeError("Social Chemistry لود نشد")

def load_scruples_records():
    url = "https://storage.googleapis.com/ai2-mosaic-public/projects/scruples/v1.0/data/anecdotes.tar.gz"
    if not os.path.exists("anecdotes"):
        urllib.request.urlretrieve(url, "anecdotes.tar.gz")
        with tarfile.open("anecdotes.tar.gz", "r:gz") as tar: tar.extractall()
    recs = []
    for sp in ["train", "dev", "test"]:
        with open(f"anecdotes/{sp}.scruples-anecdotes.jsonl") as f: recs.extend(json.loads(l) for l in f)
    print("scruples records:", len(recs)); return recs

_ST = {}
def encode_texts(texts, model_name):
    if model_name not in _ST: _ST[model_name] = SentenceTransformer(model_name, device=device)
    return _ST[model_name].encode(list(texts), batch_size=128, show_progress_bar=False,
                                  normalize_embeddings=True, convert_to_numpy=True)

def embed_cached(texts, model_name, tag):
    path = f"{CACHE}/emb_{tag}_{model_name.split('/')[-1]}.npy"
    if os.path.exists(path):
        X = np.load(path)
        if len(X) == len(texts): return X
    X = encode_texts(texts, model_name); np.save(path, X); return X

device: cuda


## 3) توابع Lens، Mapper، آزمون permutation و متریک‌ها

In [4]:
import kmapper as km, networkx as nx
from scipy.sparse import csr_matrix
from sklearn.cluster import HDBSCAN
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from statsmodels.stats.multitest import multipletests
from scipy import stats as st

def seed_of(*parts): return zlib.crc32("|".join(map(str, parts)).encode()) % (2**31)
def minmax(v):
    v = np.asarray(v, float).ravel(); return (v - v.min()) / (v.max() - v.min() + 1e-12)
def bh(p):
    p = np.asarray(p, float); out = np.full(p.shape, np.nan); ok = np.isfinite(p)
    if ok.any(): out[ok] = multipletests(p[ok], method="fdr_bh")[1]
    return out
def t_ci(x):
    x = np.asarray(pd.Series(x).dropna(), float); k = len(x)
    if k < 2: return (np.nan, np.nan, np.nan)
    h = st.t.ppf(.975, k-1) * x.std(ddof=1) / np.sqrt(k); return (x.mean(), x.mean()-h, x.mean()+h)

def chunk_dist_stats(X, k=15, chunk=1500):
    n = len(X); ecc = np.empty(n); dens = np.empty(n)
    for i in range(0, n, chunk):
        D = 1 - X[i:i+chunk] @ X.T
        ecc[i:i+chunk] = D.max(1)
        near = np.sort(np.partition(D, min(k+1, n-1), axis=1)[:, :k+1], axis=1)
        dens[i:i+chunk] = near[:, 1:k+1].mean(1)
    return ecc, dens

def make_lenses(Xtr, ytr, Xte, rng):
    L = {}
    L["pca_1d"] = minmax(PCA(1, random_state=SEED).fit(Xtr).transform(Xte))
    top2 = [c for c, _ in Counter(ytr).most_common(2)]
    ytr_a = np.array(ytr)
    d = Xtr[ytr_a == top2[1]].mean(0) - Xtr[ytr_a == top2[0]].mean(0); d /= (np.linalg.norm(d) + 1e-12)
    L["centroid_1d"] = minmax(Xte @ d)
    sims = Xte @ Xtr.T
    L["eccentricity_1d"] = minmax((1 - sims).max(1))
    L["density_1d"] = minmax(np.sort(1 - sims, axis=1)[:, :15].mean(1))
    for i in (1, 2):
        r = rng.normal(size=Xtr.shape[1]); r /= np.linalg.norm(r); L[f"random{i}_1d"] = minmax(Xte @ r)
    return L

def run_mapper(X, lens, n_cubes=N_CUBES, overlap=OVERLAP):
    return km.KeplerMapper(verbose=0).map(lens.reshape(-1, 1), X,
        cover=km.Cover(n_cubes=n_cubes, perc_overlap=overlap),
        clusterer=HDBSCAN(min_cluster_size=MIN_CLUSTER, metric="euclidean"))

def to_nx(graph):
    G = nx.Graph()
    for n, mem in graph["nodes"].items(): G.add_node(n, members=list(mem))
    for a, bs in graph["links"].items():
        for b in bs: G.add_edge(a, b)
    return G

def majority(labels):
    lab, c = Counter(labels).most_common(1)[0]; return lab, c / len(labels)

def graph_metrics(G, y_true, y_pred=None):
    y_true = np.asarray(y_true); comps = [list(c) for c in nx.connected_components(G)]
    nmaj, npur = {}, {}
    for n in G.nodes:
        nmaj[n], npur[n] = majority(y_true[G.nodes[n]["members"]])
    cp_pr, mm = [], []
    for c in comps:
        pts = sorted(set(i for n in c for i in G.nodes[n]["members"]))
        lab, _ = majority(y_true[pts])
        if y_pred is not None:
            pl, pp = majority(np.asarray(y_pred)[pts]); cp_pr.append(pp); mm.append(pl == lab)
    covered = len(set(i for n in G.nodes for i in G.nodes[n]["members"]))
    met = dict(n_nodes=G.number_of_nodes(), n_edges=G.number_of_edges(), n_components=len(comps),
               CP_Pred=(np.mean(np.array(cp_pr) >= PURE) * 100) if cp_pr else np.nan,
               MM=(np.mean(mm) * 100) if mm else np.nan,
               mixed_node=np.mean([p <= .5 for p in npur.values()]) * 100,
               noise=(1 - covered / len(y_true)) * 100)
    return met, nmaj, npur

# ----- آزمون permutation برداری‌شده روی گراف ثابت (حفظ فراوانی کلاس‌ها) -----
def prep_graph(G, n_obs):
    nodes = list(G.nodes()); pos = {n: i for i, n in enumerate(nodes)}
    r, c = [], []
    for i, n in enumerate(nodes):
        m = G.nodes[n]["members"]; r += [i]*len(m); c += list(m)
    Mnode = csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(len(nodes), n_obs))
    comps = [list(x) for x in nx.connected_components(G)]; r, c = [], []
    for i, comp in enumerate(comps):
        pts = sorted({int(p) for n in comp for p in G.nodes[n]["members"]}); r += [i]*len(pts); c += pts
    Mcomp = csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(len(comps), n_obs))
    edges = np.asarray([(pos[a], pos[b]) for a, b in G.edges()], dtype=int).reshape(-1, 2)
    return Mnode, Mcomp, edges

def _cp_ea(Mnode, Mcomp, edges, onehot_all, b, k):
    n = onehot_all.shape[0]
    cc = np.asarray(Mcomp @ onehot_all).reshape(Mcomp.shape[0], b, k); cd = np.asarray(Mcomp.sum(1)).ravel()
    v = cd > 0
    cp = 100 * np.mean((cc[v].max(2) / cd[v, None]) >= PURE, axis=0) if v.any() else np.full(b, np.nan)
    if len(edges):
        nc = np.asarray(Mnode @ onehot_all).reshape(Mnode.shape[0], b, k); nm = np.argmax(nc, axis=2)
        ea = 100 * np.mean(nm[edges[:, 0], :] == nm[edges[:, 1], :], axis=0)
    else: ea = np.full(b, np.nan)
    return cp, ea

def permutation_test_graph(G, y, n_perm, seed, batch=64):
    y = np.asarray(y); labels, codes = np.unique(y, return_inverse=True); k = len(labels); n = len(codes)
    Mnode, Mcomp, edges = prep_graph(G, n); eye = np.eye(k, dtype=np.float32)
    cp0, ea0 = _cp_ea(Mnode, Mcomp, edges, eye[codes], 1, k); cp0, ea0 = float(cp0[0]), float(ea0[0])
    rng = np.random.default_rng(seed); ncp, nea = [], []
    done = 0
    while done < n_perm:
        b = min(batch, n_perm - done)
        perms = np.vstack([rng.permutation(codes) for _ in range(b)])
        oh = eye[perms].transpose(1, 0, 2).reshape(n, b*k)
        cp, ea = _cp_ea(Mnode, Mcomp, edges, oh, b, k); ncp.append(cp); nea.append(ea); done += b
    ncp, nea = np.concatenate(ncp), np.concatenate(nea)
    def p1(obs, null):
        null = null[np.isfinite(null)]
        return (1 + np.sum(null >= obs)) / (1 + len(null)) if len(null) and np.isfinite(obs) else np.nan
    return dict(CP_GT=cp0, CP_null=np.nanmean(ncp), CP_delta=cp0 - np.nanmean(ncp), CP_p=p1(cp0, ncp),
                EA=ea0, EA_null=np.nanmean(nea) if np.isfinite(nea).any() else np.nan,
                EA_delta=ea0 - np.nanmean(nea) if np.isfinite(nea).any() else np.nan, EA_p=p1(ea0, nea))
print("helpers ready")

helpers ready


## 4) طرح اصلی (Social Chemistry، سطوح توافق): ساخت داده

In [5]:
raw = load_socchem_raw()

def build_tier_dataset(df_raw):
    df = df_raw.dropna(subset=["action", "rot", "action-moral-judgment", "action-agree", "action-pressure"]).copy()
    df = df.drop_duplicates(subset=["action"]).reset_index(drop=True)
    df["text_action"] = df["action"].astype(str).str.strip(); df["text_rot"] = df["rot"].astype(str).str.strip()
    df = df[df["text_action"].str.len() > 3].copy()
    j = df["action-moral-judgment"].astype(float); p = df["action-pressure"].astype(float); a = df["action-agree"].astype(float)
    df["judgment"] = np.where(j < 0, "wrong", np.where(j > 0, "good", "ok"))
    df["pressure"] = np.where(p < 0, "against", np.where(p > 0, "for", "none"))
    five = ["care-harm", "fairness-cheating", "loyalty-betrayal", "authority-subversion", "sanctity-degradation"]
    mf = df["rot-moral-foundations"].fillna(""); df["foundation"] = np.where(mf.isin(five), mf, "")
    df["tier"] = np.where(a <= 2, "Low", np.where(a < 4, "Mid", "High")); df["agree"] = a
    return df

df_all = build_tier_dataset(raw)
TASKS = ["judgment", "pressure", "foundation"]; TIERS = ["Low", "Mid", "High"]
print(df_all.tier.value_counts().to_dict(), df_all.judgment.value_counts().to_dict())

def sample_split(seed):
    parts = [df_all[df_all.tier == t].sample(n=min(PER_TIER_MAX, (df_all.tier == t).sum()), random_state=seed) for t in TIERS]
    df = pd.concat(parts, ignore_index=True)
    tr, te = train_test_split(np.arange(len(df)), test_size=TEST_FRAC, random_state=seed, stratify=df["tier"] + "_" + df["judgment"])
    df["split"] = "train"; df.loc[te, "split"] = "test"; return df

DF = {s: sample_split(s) for s in TIER_SEEDS}
t1 = (DF[SEED].groupby(["split", "tier"]).agg(n=("text_action", "size"),
      wrong=("judgment", lambda s: round((s == "wrong").mean()*100, 1)), good=("judgment", lambda s: round((s == "good").mean()*100, 1)),
      pressure_for=("pressure", lambda s: round((s == "for").mean()*100, 1)), pressure_against=("pressure", lambda s: round((s == "against").mean()*100, 1))).reset_index())
t1.to_csv(f"{OUT}/table1_dataset_stats.csv", index=False); t1

social-chem-101.v1.0.tsv: reconstructing file:   0%|          |  0.00B /  147MB            

social-chem-101.v1.0.tsv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/355922 [00:00<?, ? examples/s]

loaded: tasksource/social-chemestry-101 (355922, 25)
{'Mid': 121538, 'High': 64434, 'Low': 39085} {'ok': 104482, 'wrong': 73993, 'good': 46582}


,split,tier,n,wrong,good,pressure_for,pressure_against
0,test,High,1200,30.2,24.4,42.2,29.2
1,test,Low,1200,40.3,11.8,8.2,31.1
2,test,Mid,1200,32.9,20.5,33.1,31.8
3,train,High,2800,30.3,24.4,42.2,30.1
4,train,Low,2800,40.3,11.7,9.2,33.0
5,train,Mid,2800,33.0,20.5,32.0,31.6


## 5) اجرای Mapper + آزمون‌های permutation (همه‌ی مدل‌ها در seed اصلی؛ مدل اصلی در seedهای تکراری)

In [6]:
def run_tier_design(df, X, mshort, seed, n_perm, keep=None, text_tag=""):
    rows = []
    for task in TASKS:
        use = (df[task] != "").values
        tr = np.where((df.split == "train").values & use)[0]; te_all = np.where((df.split == "test").values & use)[0]
        clf = LogisticRegression(max_iter=1000).fit(X[tr], df[task].values[tr])
        for tier in TIERS:
            te = te_all[df.tier.values[te_all] == tier]
            if len(te) < 50: continue
            Xte, yte = X[te], df[task].values[te]; ypr = clf.predict(Xte)
            lenses = make_lenses(X[tr], df[task].values[tr], Xte, np.random.default_rng(seed))
            for lname, lens in lenses.items():
                G = to_nx(run_mapper(Xte, lens))
                if G.number_of_nodes() < 2: continue
                met, nmaj, npur = graph_metrics(G, yte, ypr)
                test = permutation_test_graph(G, yte, n_perm, seed_of(mshort, task, tier, lname, seed, text_tag))
                rows.append(dict(model=mshort, seed=seed, text=text_tag or "action", task=task, tier=tier, lens=lname, n=len(te), **met, **test))
                if keep is not None: keep[(task, tier, lname)] = dict(G=G, nmaj=nmaj, npur=npur, te=te, yte=yte, lens=lens, Xte=Xte)
    return rows

t0 = time.time(); main_rows = []; KEEP = {}
for m in MODELS:
    ms = m.split("/")[-1]; X = embed_cached(DF[SEED].text_action.tolist(), m, f"tier{SEED}_action")
    main_rows += run_tier_design(DF[SEED], X, ms, SEED, N_PERM_MAIN, keep=KEEP if ms == PRIMARY else None)
    print(ms, "done", f"{time.time()-t0:.0f}s")
main = pd.DataFrame(main_rows)
main["CP_q"] = bh(main.CP_p); main["EA_q"] = bh(main.EA_p)        # BH روی همه‌ی آزمون‌ها
main.to_csv(f"{OUT}/table2_main_graphs_formal_tests.csv", index=False)

seed_rows = []                                                      # تکرار seed ها (مدل اصلی)
for s in TIER_SEEDS:
    if s == SEED:
        seed_rows += [r for r in main_rows if r["model"] == PRIMARY]; continue
    X = embed_cached(DF[s].text_action.tolist(), MODELS[0], f"tier{s}_action")
    seed_rows += run_tier_design(DF[s], X, PRIMARY, s, N_PERM_SEED)
    print("seed", s, "done", f"{time.time()-t0:.0f}s")
seeds_df = pd.DataFrame(seed_rows); seeds_df.to_csv(f"{OUT}/table3_tier_design_by_seed_primary.csv", index=False)

# حذف نشت واژگانی: همان نمونه با متن rule-of-thumb
X = embed_cached(DF[SEED].text_rot.tolist(), MODELS[0], f"tier{SEED}_rot")
abl = pd.DataFrame(run_tier_design(DF[SEED].assign(text_action=DF[SEED].text_rot), X, PRIMARY, SEED, N_PERM_ABL, text_tag="rot"))
abl.to_csv(f"{OUT}/table4_text_field_ablation_rot.csv", index=False)
print("main", main.shape, "seeds", seeds_df.shape, "ablation", abl.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (32 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

all-mpnet-base-v2 done 137s
seed 43 done 163s
seed 44 done 189s
seed 45 done 214s
seed 46 done 239s
main (162, 24) seeds (270, 22) ablation (54, 22)


## 6) H3 — پیش‌بینی نسبت به baseline و رابطه‌ی توافق با صحت

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score
import statsmodels.api as sm, statsmodels.formula.api as smf

def boot_delta(y, p, b, n=N_BOOT, seed=0):
    rng = np.random.default_rng(seed); y = np.asarray(y); d = []
    for _ in range(n):
        ix = rng.integers(0, len(y), len(y)); d.append(balanced_accuracy_score(y[ix], p[ix]) - balanced_accuracy_score(y[ix], b[ix]))
    return np.quantile(d, [.025, .975])

pred_rows, glm_rows = [], []; df0 = DF[SEED]
for m in MODELS:
    ms = m.split("/")[-1]; X = embed_cached(df0.text_action.tolist(), m, f"tier{SEED}_action")
    for task in TASKS:
        use = (df0[task] != "").values
        tr = np.where((df0.split == "train").values & use)[0]; te_all = np.where((df0.split == "test").values & use)[0]
        ytr = df0[task].values[tr]
        lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X[tr], ytr)
        d_maj = DummyClassifier(strategy="most_frequent").fit(X[tr], ytr)
        d_str = DummyClassifier(strategy="stratified", random_state=SEED).fit(X[tr], ytr)
        for tier in TIERS:
            te = te_all[df0.tier.values[te_all] == tier]
            if len(te) < 30: continue
            y = df0[task].values[te]; p = lr.predict(X[te]); bm = d_maj.predict(X[te]); bs = d_str.predict(X[te])
            lo_m, _ = boot_delta(y, p, bm, seed=seed_of(ms, task, tier, "m")); lo_s, _ = boot_delta(y, p, bs, seed=seed_of(ms, task, tier, "s"))
            pred_rows.append(dict(model=ms, task=task, tier=tier, n=len(te), acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
                macro_f1=f1_score(y, p, average="macro"), bal_acc_majority=balanced_accuracy_score(y, bm), bal_acc_stratified=balanced_accuracy_score(y, bs),
                delta_vs_majority_CIlow=lo_m, delta_vs_stratified_CIlow=lo_s, passes=bool(lo_m > 0 and lo_s > 0)))
        yA = df0[task].values[te_all]; pA = lr.predict(X[te_all]); ag = df0["agree"].values[te_all]
        g = pd.DataFrame(dict(correct=(yA == pA).astype(int), agree=ag, true_label=yA.astype(str)))
        try:
            fit = smf.glm("correct ~ agree + C(true_label)", data=g, family=sm.families.Binomial()).fit(cov_type="HC3")
            lo, hi = fit.conf_int().loc["agree"]; glm_rows.append(dict(model=ms, task=task, n=len(g), OR=float(np.exp(fit.params["agree"])),
                OR_low=float(np.exp(lo)), OR_high=float(np.exp(hi)), p=float(fit.pvalues["agree"])))
        except Exception as e: glm_rows.append(dict(model=ms, task=task, n=len(g), OR=np.nan, p=np.nan, error=str(e)[:100]))
pred = pd.DataFrame(pred_rows); glm = pd.DataFrame(glm_rows); glm["q"] = bh(glm.p)
pred.to_csv(f"{OUT}/table5_prediction_vs_baselines.csv", index=False); glm.to_csv(f"{OUT}/table6_agreement_vs_correctness_GLM.csv", index=False)
print(pred.passes.mean(), (glm.q < .05).mean())

1.0 0.8888888888888888


## 7) H6 — حساسیت به پارامترهای Mapper (مدل اصلی، شبکه‌ی ۴×۳، سه lens)

In [8]:
sens_rows = []; cnt = 0
for (task, tier, lname), v in KEEP.items():
    if lname not in ("pca_1d", "eccentricity_1d", "density_1d"): continue
    for nc in [20, 30, 40, 50]:
        for ov in [0.2, 0.3, 0.4]:
            cnt += 1
            try:
                G = to_nx(run_mapper(v["Xte"], v["lens"], nc, ov))
                if G.number_of_nodes() < 2: continue
                t = permutation_test_graph(G, v["yte"], N_PERM_SENS, seed_of("sens", task, tier, lname, nc, ov))
                sens_rows.append(dict(task=task, tier=tier, lens=lname, n_cubes=nc, overlap=ov, n_nodes=G.number_of_nodes(), **t))
            except Exception as e: print("sens error", task, tier, lname, nc, ov, str(e)[:80])
sens = pd.DataFrame(sens_rows); sens.to_csv(f"{OUT}/table7_mapper_parameter_grid.csv", index=False); print("grid configs:", len(sens))

grid configs: 324


## 8) H7 — اختلاف واقعی annotator ها: Social Chemistry (چند annotator) و Scruples (توزیع رأی)

In [9]:
def metrics_real(nodes, links, y, valid, ent=None):
    maj, pur = {}, {}
    for k, idx in nodes.items():
        ii = [i for i in idx if valid[i]]
        if not ii: continue
        l, c = Counter(y[ii]).most_common(1)[0]; maj[k] = l; pur[k] = c / len(ii)
    G = nx.Graph(); G.add_nodes_from(maj)
    for a, bs in links.items():
        for b in bs:
            if a in maj and b in maj: G.add_edge(a, b)
    ea = np.mean([maj[a] == maj[b] for a, b in G.edges]) if G.number_of_edges() else np.nan
    cps, sizes = [], []
    for comp in nx.connected_components(G):
        pts = sorted({i for k in comp for i in nodes[k] if valid[i]})
        if pts: cps.append(Counter(y[pts]).most_common(1)[0][1] / len(pts)); sizes.append(len(pts))
    cps, sizes = np.array(cps), np.array(sizes)
    out = dict(CP=100*np.mean(cps >= PURE), EA=100*ea, mixed=100*np.mean([p <= .5 for p in pur.values()]), n_nodes=len(maj))
    if ent is not None:
        big = [k for k in maj if sum(valid[i] for i in nodes[k]) >= 5]
        if len(big) > 10 and np.ptp([np.mean(ent[nodes[k]]) for k in big]) > 0:
            out["rho"] = st.spearmanr([np.mean(ent[nodes[k]]) for k in big], [1 - pur[k] for k in big])[0]
    return out

def excess_real(nodes, links, y, valid, rng, ent=None):
    real = metrics_real(nodes, links, y, valid, ent); vi = np.where(valid)[0]; P = []
    for _ in range(N_PERM_REAL):
        ys = y.copy(); ys[vi] = rng.permutation(y[vi]); P.append(metrics_real(nodes, links, ys, valid))
    for k in ("CP", "EA"): real["d" + k] = real[k] - np.nanmean([p[k] for p in P])
    return real

def lenses_real(X):
    ecc, dens = chunk_dist_stats(X)
    return {"pca": minmax(PCA(1, random_state=0).fit_transform(X)), "ecc": minmax(ecc), "dens": minmax(dens)}

def build_real(X, lens):
    g = run_mapper(X, lens); return g["nodes"], g["links"]

def run_real(R, E, seeds, tag):
    y_all = R.label.values.astype(str); valid_all = ~R.tie.values; ent_all = R.ent.values
    GROUPS = ["Unanimous", "Mild", "Strong"]; m = int(R.group.value_counts().min()); out = []
    for seed in seeds:
        rng = np.random.default_rng(seed)
        pick = {g: rng.choice(R.index[R.group == g], m, replace=False) for g in GROUPS}
        pick["POOLED"] = np.concatenate([pick[g] for g in GROUPS])
        for mname, Xa in E.items():
            for g, idx in pick.items():
                X = Xa[idx]
                for ln, lens in lenses_real(X).items():
                    nodes, links = build_real(X, lens)
                    out.append(dict(dataset=tag, seed=seed, model=mname, group=g, lens=ln, **excess_real(nodes, links, y_all[idx], valid_all[idx], rng, ent=ent_all[idx])))
        print(tag, "seed", seed, "done")
    return pd.DataFrame(out)

# ---- Social Chemistry: RoT هایی با دقیقاً ۶ قضاوت مستقل ----
d = raw.dropna(subset=["rot-id", "action-moral-judgment", "action-agree"]).copy()
cnt_rot = d.groupby("rot-id").size(); d6 = d[d["rot-id"].isin(cnt_rot[cnt_rot == 6].index)].copy()
lab = lambda x: "wrong" if x < 0 else ("good" if x > 0 else "ok")
d6["lab"] = d6["action-moral-judgment"].map(lab); rows = []
for rid, g in d6.groupby("rot-id"):
    c = Counter(g.lab).most_common(); p = np.array([v for _, v in c]) / len(g)
    rows.append(dict(rot=g.rot.iloc[0], ent=float(-(p*np.log2(p)).sum()), tie=(len(c) > 1 and c[0][1] == c[1][1]), label=c[0][0], top=c[0][1]))
R_sc = pd.DataFrame(rows); R_sc["group"] = np.where(R_sc.top == 6, "Unanimous", np.where(R_sc.top == 5, "Mild", "Strong"))
print(pd.crosstab(R_sc.group, R_sc.label))
E_sc = {m.split("/")[-1]: embed_cached(R_sc.rot.tolist(), m, "socchem_multi") for m in MODELS}
real_sc = run_real(R_sc, E_sc, REAL_SEEDS, "SocialChem_6annotators"); real_sc.to_csv(f"{OUT}/table8_real_disagreement_socialchem.csv", index=False)

# ---- Scruples: توزیع رأی جامعه (۸ رأی هم‌اندازه با نمونه‌گیری hypergeometric) ----
CLASSES = ["AUTHOR", "OTHER", "EVERYBODY", "NOBODY"]; K = 8
recs = load_scruples_records(); rng0 = np.random.default_rng(0); rows = []
for r in recs:
    s = r.get("label_scores") or r.get("scores"); c = np.array([s.get(k, 0) for k in CLASSES], dtype=int)
    a = r.get("action"); t = (a.get("description") if isinstance(a, dict) else a) or r.get("title")
    if not t or c.sum() < K: continue
    c8 = rng0.multivariate_hypergeometric(c, K); p = c8[c8 > 0] / K; top = np.sort(c8)[::-1]
    rows.append(dict(rot=t, ent=float(-(p*np.log2(p)).sum()), tie=bool(top[0] == top[1]), label=CLASSES[int(c8.argmax())]))
R_sr = pd.DataFrame(rows); R_sr["group"] = np.where(R_sr.ent < 1e-9, "Unanimous", np.where(R_sr.ent <= 0.82, "Mild", "Strong"))
print(len(R_sr), R_sr.group.value_counts().to_dict())
E_sr = {m.split("/")[-1]: embed_cached(R_sr.rot.tolist(), m, "scruples") for m in MODELS}
real_sr = run_real(R_sr, E_sr, REAL_SEEDS, "Scruples_votes"); real_sr.to_csv(f"{OUT}/table9_real_disagreement_scruples.csv", index=False)
real = pd.concat([real_sc, real_sr], ignore_index=True)

label      good   ok  wrong
group                      
Mild        285  533    228
Strong      518  615     71
Unanimous   314  903   1752
SocialChem_6annotators seed 1 done
SocialChem_6annotators seed 2 done
SocialChem_6annotators seed 3 done
SocialChem_6annotators seed 4 done
SocialChem_6annotators seed 5 done
scruples records: 32766
16196 {'Strong': 7701, 'Mild': 4753, 'Unanimous': 3742}
Scruples_votes seed 1 done
Scruples_votes seed 2 done
Scruples_votes seed 3 done
Scruples_votes seed 4 done
Scruples_votes seed 5 done


## 9) سازگاری بین مدل‌ها (H4)، lens (H5)، pressure در برابر judgment (H9)

In [10]:
from scipy.stats import friedmanchisquare, spearmanr, kendalltau, wilcoxon
inf = main[main.lens.isin(INFORMATIVE)].copy(); models = [m.split("/")[-1] for m in MODELS]

# H4
h4 = []
for metric in ["CP_delta", "EA_delta"]:
    piv = inf.pivot_table(index=["task", "tier", "lens"], columns="model", values=metric)
    for a, b in combinations(models, 2):
        pr = piv[[a, b]].dropna()
        rho = spearmanr(pr[a], pr[b]).statistic; rng = np.random.default_rng(seed_of("h4", metric, a, b)); bs = []
        for _ in range(1500):
            ix = rng.integers(0, len(pr), len(pr)); x, y = pr[a].values[ix], pr[b].values[ix]
            if np.ptp(x) > 0 and np.ptp(y) > 0: bs.append(spearmanr(x, y).statistic)
        lo = np.quantile(bs, .025) if len(bs) > 30 else np.nan
        h4.append(dict(metric=metric, m1=a, m2=b, n=len(pr), rho=rho, rho_CIlow=lo, same_direction=float(((pr[a] > 0) == (pr[b] > 0)).mean())))
h4 = pd.DataFrame(h4); h4.to_csv(f"{OUT}/table10_cross_embedding.csv", index=False)

# H5
h5 = []
for m in models:
    for metric in ["CP_delta", "EA_delta"]:
        piv = inf[inf.model == m].pivot_table(index=["task", "tier"], columns="lens", values=metric).dropna()
        if len(piv) >= 5 and piv.shape[1] >= 3:
            s, p = friedmanchisquare(*[piv[c].values for c in piv.columns]); h5.append(dict(model=m, metric=metric, n_blocks=len(piv), chi2=s, p=p, kendall_W=s / (len(piv) * (piv.shape[1] - 1))))
h5 = pd.DataFrame(h5); h5["q"] = bh(h5.p); h5.to_csv(f"{OUT}/table11_lens_invariance.csv", index=False)

# H9
h9 = []
for metric in ["CP_delta", "EA_delta"]:
    piv = inf[inf.tier.isin(["High", "Mid"])].pivot_table(index=["model", "tier", "lens"], columns="task", values=metric).dropna(subset=["judgment", "pressure"])
    d_ = (piv["pressure"] - piv["judgment"]).values
    try: p_less = wilcoxon(d_, alternative="less").pvalue; p_greater = wilcoxon(d_, alternative="greater").pvalue
    except Exception: p_less = p_greater = np.nan
    h9.append(dict(metric=metric, n_pairs=len(d_), median_pressure_minus_judgment=float(np.median(d_)), p_pressure_more_interwoven=p_less, p_pressure_less_interwoven=p_greater))
h9 = pd.DataFrame(h9); h9["q_more_interwoven"] = bh(h9.p_pressure_more_interwoven); h9.to_csv(f"{OUT}/table12_pressure_vs_judgment.csv", index=False)
print(h4.round(2)); print(h5.round(3)); print(h9.round(3))

     metric                 m1                 m2   n   rho  rho_CIlow  \
0  CP_delta   all-MiniLM-L6-v2  bge-small-en-v1.5  36  0.42       0.10   
1  CP_delta   all-MiniLM-L6-v2  all-mpnet-base-v2  36  0.55       0.26   
2  CP_delta  bge-small-en-v1.5  all-mpnet-base-v2  36  0.62       0.33   
3  EA_delta   all-MiniLM-L6-v2  bge-small-en-v1.5  36  0.69       0.46   
4  EA_delta   all-MiniLM-L6-v2  all-mpnet-base-v2  36  0.52       0.22   
5  EA_delta  bge-small-en-v1.5  all-mpnet-base-v2  36  0.61       0.33   

   same_direction  
0            0.94  
1            0.97  
2            0.97  
3            0.72  
4            0.64  
5            0.69  
               model    metric  n_blocks    chi2      p  kendall_W      q
0   all-MiniLM-L6-v2  CP_delta         9   6.733  0.081      0.249  0.205
1   all-MiniLM-L6-v2  EA_delta         9   6.200  0.102      0.230  0.205
2  bge-small-en-v1.5  CP_delta         9   3.267  0.352      0.121  0.352
3  bge-small-en-v1.5  EA_delta         9   3.

## 10) حکم نهایی همه‌ی فرضیه‌ها (قواعد ثابت)

In [11]:
A = RULES["ALPHA"]; V = []
def add(hid, claim, rule, evidence, ok): V.append(dict(ID=hid, claim=claim, rule=rule, evidence=evidence, VERDICT="SUPPORTED" if ok else "NOT SUPPORTED"))

# ---- H1 / H2 ----
def cond_pass(metric, q):
    rows = []
    for (m, t, ti), g in inf.groupby(["model", "task", "tier"]):
        rows.append(dict(model=m, task=t, tier=ti, ok=int(((g[metric] > 0) & (g[q] < A)).sum()) >= RULES["H1H2_LENS_MAJORITY"]))
    return pd.DataFrame(rows)
for hid, metric, q, name in [("H1", "CP_delta", "CP_q", "Component purity > shuffled-label chance"), ("H2", "EA_delta", "EA_q", "Edge agreement > shuffled-label chance")]:
    cp = cond_pass(metric, q); by_model = cp.groupby("model").ok.mean(); by_tier = cp.groupby("tier").ok.mean().round(2).to_dict()
    cp.to_csv(f"{OUT}/verdict_{hid}_conditions.csv", index=False)
    add(hid, name, f"each model: >= {RULES['H1H2_COND_FRACTION']:.2f} of 9 task×tier conditions have >=3/4 informative lenses with positive effect and BH q<.05",
        f"pass fraction by model={by_model.round(2).to_dict()}; by tier={by_tier}", bool((by_model >= RULES["H1H2_COND_FRACTION"]).all() and len(by_model) == len(models)))

# ---- H3 ----
add("H3a", "Linear probe beats majority & stratified baselines", f">= {RULES['H3A_FRACTION']:.0%} of model×task×tier cells: balanced-accuracy gain with bootstrap CI > 0 vs both baselines",
    f"pass fraction={pred.passes.mean():.2f} ({int(pred.passes.sum())}/{len(pred)})", bool(pred.passes.mean() >= RULES["H3A_FRACTION"]))
g_ok = ((glm.q < A) & (glm.OR > 1)); add("H3b", "Higher annotator agreement -> higher probe correctness", f">= {RULES['H3B_FRACTION']:.2f} of model×task: OR>1 and BH q<.05 (class-adjusted GLM)",
    f"significant positive={int(g_ok.sum())}/{len(glm)}; median OR={glm.OR.median():.3f}", bool(g_ok.mean() >= RULES["H3B_FRACTION"]))

# ---- H4 ----
ok4 = bool(((h4.rho >= RULES["H4_RHO_MIN"]) & (h4.rho_CIlow > 0) & (h4.same_direction >= RULES["H4_SAMEDIR_MIN"])).all())
add("H4", "Findings are consistent across the 3 embedding models", f"all 3 model pairs × 2 metrics: Spearman rho>={RULES['H4_RHO_MIN']} (CI low>0) and same-direction>={RULES['H4_SAMEDIR_MIN']}",
    f"rho range={h4.rho.min():.2f}..{h4.rho.max():.2f}; same-dir min={h4.same_direction.min():.2f}", ok4)

# ---- H5 ----
ok5 = bool(((h5.q >= A) & (h5.kendall_W < RULES["H5_W_MAX"])).all()) if len(h5) else False
add("H5", "Results are lens-invariant (4 informative lenses)", f"all model×metric Friedman tests: q>=.05 and Kendall W<{RULES['H5_W_MAX']}",
    f"min q={h5.q.min():.3f}; max W={h5.kendall_W.max():.2f}", ok5)

# ---- H6 ----
pc, pe = float((sens.CP_delta > 0).mean()), float((sens.EA_delta > 0).mean())
add("H6", "Positive effects are stable across Mapper parameters (r, overlap)", f">= {RULES['H6_POS_FRACTION']:.0%} of grid configs positive for BOTH CP and EA",
    f"CP positive={pc:.2f}; EA positive={pe:.2f}; n={len(sens)}", bool(pc >= RULES["H6_POS_FRACTION"] and pe >= RULES["H6_POS_FRACTION"]))

# ---- H7a/b/c : اختلاف واقعی annotator ها ----
ps = real.groupby(["dataset", "model", "group", "seed"])[["dCP", "dEA", "rho"]].mean().reset_index(); rows7 = []
for (ds, mo), g in ps.groupby(["dataset", "model"]):
    pv = g.pivot(index="seed", columns="group", values=["dCP", "dEA", "rho"])
    r = dict(dataset=ds, model=mo)
    for met in ("dCP", "dEA"):
        mean, lo, hi = t_ci(pv[met]["Unanimous"] - pv[met]["Strong"]); r[f"{met}_Unan_minus_Strong"] = mean; r[f"{met}_CIlow"] = lo
        r[f"{met}_Unan"] = pv[met]["Unanimous"].mean(); r[f"{met}_Mild"] = pv[met]["Mild"].mean(); r[f"{met}_Strong"] = pv[met]["Strong"].mean()
    m_, lo_, hi_ = t_ci(pv["rho"]["POOLED"]); r["rho_pooled"] = m_; r["rho_CIlow"] = lo_; rows7.append(r)
h7 = pd.DataFrame(rows7); h7.to_csv(f"{OUT}/table13_real_disagreement_summary.csv", index=False)
add("H7a", "Real annotator disagreement lowers component-purity excess (Unanimous > Strong)", "both datasets × all 3 models: paired seed-level CI of (Unanimous-Strong) dCP > 0",
    "; ".join(f"{r.dataset[:8]}/{r.model[:8]}: {r.dCP_Unan_minus_Strong:+.1f} (CIlow {r.dCP_CIlow:+.1f})" for r in h7.itertuples()), bool((h7.dCP_CIlow > 0).all()))
add("H7b", "Real annotator disagreement lowers edge-agreement excess (Unanimous > Strong)", "both datasets × all 3 models: paired seed-level CI of (Unanimous-Strong) dEA > 0",
    "; ".join(f"{r.dataset[:8]}/{r.model[:8]}: {r.dEA_Unan_minus_Strong:+.1f} (CIlow {r.dEA_CIlow:+.1f})" for r in h7.itertuples()), bool((h7.dEA_CIlow > 0).all()))
add("H7c", "Nodes with higher disagreement are more impure (rho>0)", "both datasets × all 3 models: pooled Spearman rho seed-level CI low > 0",
    "; ".join(f"{r.dataset[:8]}/{r.model[:8]}: rho={r.rho_pooled:.2f} (CIlow {r.rho_CIlow:+.2f})" for r in h7.itertuples()), bool((h7.rho_CIlow > 0).all()))

# ---- H7d : پراکسی سطح توافق، seed های تکراری ----
sd = seeds_df[seeds_df.lens.isin(INFORMATIVE)].groupby(["seed", "task", "tier"])[["CP_delta", "EA_delta"]].mean().reset_index(); rows = []
for task in TASKS:
    r = dict(task=task)
    for met in ("CP_delta", "EA_delta"):
        pv = sd[sd.task == task].pivot(index="seed", columns="tier", values=met); mean, lo, hi = t_ci(pv["High"] - pv["Low"]); r[met + "_High_minus_Low"] = mean; r[met + "_CIlow"] = lo
    rows.append(r)
h7d = pd.DataFrame(rows); h7d["passes_both"] = (h7d.CP_delta_CIlow > 0) & (h7d.EA_delta_CIlow > 0); h7d.to_csv(f"{OUT}/table14_tier_proxy_by_seed.csv", index=False)
add("H7d", "Agreement-tier proxy: High-agreement tier has larger purity & edge excess than Low", f">= {RULES['H7D_TASK_MIN']} of 3 tasks: seed-level CI of (High-Low) > 0 for CP and EA (primary model, {len(TIER_SEEDS)} seeds)",
    "; ".join(f"{r.task}: CP {r.CP_delta_High_minus_Low:+.1f} (CIlow {r.CP_delta_CIlow:+.1f}), EA {r.EA_delta_High_minus_Low:+.1f} (CIlow {r.EA_delta_CIlow:+.1f})" for r in h7d.itertuples()),
    bool(h7d.passes_both.sum() >= RULES["H7D_TASK_MIN"]))

# ---- H8 : نشت واژگانی ----
ab = abl[abl.lens.isin(INFORMATIVE)]; posf = float((ab.CP_delta > 0).mean())
ordok = 0
for task in TASKS:
    g = ab[ab.task == task].groupby("tier").CP_delta.mean()
    if {"High", "Low"} <= set(g.index) and g["High"] > g["Low"]: ordok += 1
add("H8", "Conclusions hold when embedding the rule-of-thumb text instead of the action phrase", f">= {RULES['H8_POS_FRACTION']:.0%} informative cells with positive CP excess and High>Low ordering in >= {RULES['H8_TASK_MIN']} tasks",
    f"positive CP cells={posf:.2f}; tasks with High>Low={ordok}/3", bool(posf >= RULES["H8_POS_FRACTION"] and ordok >= RULES["H8_TASK_MIN"]))

# ---- H9 : pressure vs judgment (فرضیه H3 قدیمیِ مقاله) ----
ok9 = bool((h9.q_more_interwoven < A).all() and (h9.median_pressure_minus_judgment < 0).all())
add("H9", "Social-pressure labels are more interwoven than moral-judgment labels (lower excess)", "High+Mid tiers, informative lenses, all models: one-sided Wilcoxon (pressure<judgment) BH q<.05 for CP and EA",
    "; ".join(f"{r.metric}: median(pressure-judgment)={r.median_pressure_minus_judgment:+.1f}, p(more interwoven)={r.p_pressure_more_interwoven:.3f}, p(less interwoven)={r.p_pressure_less_interwoven:.3f}" for r in h9.itertuples()), ok9)

verdicts = pd.DataFrame(V); verdicts.to_csv(f"{OUT}/FINAL_VERDICTS.csv", index=False)
pd.set_option("display.max_colwidth", 400, "display.width", 250); print(verdicts[["ID", "claim", "VERDICT"]].to_string(index=False))

 ID                                                                                claim       VERDICT
 H1                                             Component purity > shuffled-label chance NOT SUPPORTED
 H2                                               Edge agreement > shuffled-label chance NOT SUPPORTED
H3a                                   Linear probe beats majority & stratified baselines     SUPPORTED
H3b                               Higher annotator agreement -> higher probe correctness     SUPPORTED
 H4                                Findings are consistent across the 3 embedding models NOT SUPPORTED
 H5                                    Results are lens-invariant (4 informative lenses) NOT SUPPORTED
 H6                    Positive effects are stable across Mapper parameters (r, overlap) NOT SUPPORTED
H7a      Real annotator disagreement lowers component-purity excess (Unanimous > Strong)     SUPPORTED
H7b        Real annotator disagreement lowers edge-agreement excess (Unan

## 11) شکل‌ها، گزارش و دانلود

In [12]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
# شکل A: اثر (real - shuffle) به تفکیک tier و تسک، مدل اصلی، میانگین seed ها ± CI
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, met, ttl in zip(axes, ["CP_delta", "EA_delta"], ["Pure components: real - shuffled (pts)", "Edge agreement: real - shuffled (pts)"]):
    for i, tier in enumerate(["High", "Mid", "Low"]):
        mu, err = [], []
        for task in TASKS:
            pv = sd[(sd.task == task)].pivot(index="seed", columns="tier", values=met)[tier]; m_, lo_, hi_ = t_ci(pv); mu.append(m_); err.append(m_ - lo_ if np.isfinite(lo_) else 0)
        ax.bar(np.arange(3) + (i-1)*0.26, mu, 0.26, yerr=err, capsize=2, label=tier)
    ax.axhline(0, color="k", lw=.8); ax.set_xticks(range(3)); ax.set_xticklabels(TASKS); ax.set_title(ttl, fontsize=10); ax.legend(title="agreement tier", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_A_tier_effects_with_seed_CI.png", dpi=200); plt.close()

# شکل B: اختلاف واقعی annotator ها
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5))
for r_i, ds in enumerate(["SocialChem_6annotators", "Scruples_votes"]):
    for c_i, met in enumerate(["dCP", "dEA"]):
        ax = axes[r_i, c_i]; sub = ps[ps.dataset == ds]
        for j, grp in enumerate(["Unanimous", "Mild", "Strong"]):
            mu, err = [], []
            for mo in models:
                m_, lo_, hi_ = t_ci(sub[(sub.model == mo) & (sub.group == grp)][met]); mu.append(m_); err.append(m_ - lo_ if np.isfinite(lo_) else 0)
            ax.bar(np.arange(3) + (j-1)*0.26, mu, 0.26, yerr=err, capsize=2, label=grp)
        ax.axhline(0, color="k", lw=.8); ax.set_xticks(range(3)); ax.set_xticklabels([m[:14] for m in models], fontsize=8); ax.set_title(f"{ds}: {met} (real - shuffled)", fontsize=9)
        if r_i == 0 and c_i == 0: ax.legend(title="annotator disagreement", fontsize=7)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_B_real_disagreement.png", dpi=200); plt.close()

# شکل C: observed در برابر null (مدل اصلی)
pl = main[(main.model == PRIMARY) & (main.lens.isin(INFORMATIVE))].groupby(["task", "tier"])[["CP_GT", "CP_null", "EA", "EA_null"]].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (a, b, t) in zip(axes, [("CP_GT", "CP_null", "Pure components (%)"), ("EA", "EA_null", "Edge agreement (%)")]):
    x = np.arange(len(pl)); ax.bar(x-.2, pl[a], .4, label="real"); ax.bar(x+.2, pl[b], .4, label="shuffled (999 perms)")
    ax.set_xticks(x); ax.set_xticklabels(pl.task.str[:5] + "\n" + pl.tier, fontsize=7); ax.set_title(t); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig_C_real_vs_null.png", dpi=200); plt.close()

lines = ["# FINAL VERDICTS", "", "Rules were fixed before the run (see RULES in the config cell). VERDICT is deterministic given the data.", ""]
for r in verdicts.itertuples():
    lines += [f"## {r.ID}: {r.VERDICT}", f"- Claim: {r.claim}", f"- Rule: {r.rule}", f"- Evidence: {r.evidence}", ""]
lines += ["## Scope and caveats (always report these)",
 "- Verdicts cover Social Chemistry 101 (actions/rules, English, Reddit-derived) and Scruples; 3 small/medium frozen encoders; Mapper settings of Rair et al. Not causal, not general to other corpora or languages.",
 "- Social Chemistry `action-agree` is one annotator's estimate (proxy). Real disagreement is measured only in H7a-c (6-annotator RoTs; Scruples votes subsampled to 8).",
 "- Seed-level CIs reflect re-sampling of the same finite corpora, not new datasets. Lens variants are related views of the same data, not independent replications."]
open(f"{OUT}/FINAL_REPORT.md", "w", encoding="utf-8").write("\n".join(lines))
import shutil; shutil.make_archive("final_results", "zip", OUT)
print(open(f"{OUT}/FINAL_REPORT.md", encoding="utf-8").read()[:6000])
try:
    from google.colab import files; files.download("final_results.zip")
except Exception: print("final_results.zip ساخته شد؛ از پنل Files دانلود کن.")

# FINAL VERDICTS

Rules were fixed before the run (see RULES in the config cell). VERDICT is deterministic given the data.

## H1: NOT SUPPORTED
- Claim: Component purity > shuffled-label chance
- Rule: each model: >= 0.67 of 9 task×tier conditions have >=3/4 informative lenses with positive effect and BH q<.05
- Evidence: pass fraction by model={'all-MiniLM-L6-v2': 0.11, 'all-mpnet-base-v2': 0.56, 'bge-small-en-v1.5': 0.44}; by tier={'High': 0.67, 'Low': 0.0, 'Mid': 0.44}

## H2: NOT SUPPORTED
- Claim: Edge agreement > shuffled-label chance
- Rule: each model: >= 0.67 of 9 task×tier conditions have >=3/4 informative lenses with positive effect and BH q<.05
- Evidence: pass fraction by model={'all-MiniLM-L6-v2': 0.11, 'all-mpnet-base-v2': 0.22, 'bge-small-en-v1.5': 0.22}; by tier={'High': 0.33, 'Low': 0.0, 'Mid': 0.22}

## H3a: SUPPORTED
- Claim: Linear probe beats majority & stratified baselines
- Rule: >= 90% of model×task×tier cells: balanced-accuracy gain with bootstrap CI > 0 vs b

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>